# 00 Daily tables: getting the data ready

This notebook builds (and later tops up) the **daily tables**: the S3 landing data summarised once per landing day, for every brand at once. Features, labels and brand lookups read only these tables; a missing day is built automatically the first time it is needed.

How it works (`src/features/daily_tables.py`):

1. Every S3 file of a day is read **in parallel**, and **only the columns the project uses**. Parquet lets you fetch column chunks alone, and those columns are about 7% of the bytes, so nothing is fully downloaded and the personal data columns never leave S3.
2. The rows are aggregated in memory to **one row per brand, player and day**, and only that is written: about 1.3 MB per day for both operators. No raw file ever touches the disk.
3. The daily profile used by the data alerts is computed in the same pass.
4. A day that is already built is skipped, so a later run only reads the new days.

Every step below is timed.

In [1]:
import sys, time, datetime as dt, shutil
from contextlib import contextmanager
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src" / "features"))
import daily_tables
import landing as bf  # landing helpers (the daily tables are landing)

TIMINGS = {}

@contextmanager
def timer(name):
    start = time.perf_counter()
    yield
    TIMINGS[name] = time.perf_counter() - start
    print(f"[timer] {name}: {TIMINGS[name]:.1f} s ({TIMINGS[name] / 60:.1f} min)")

START, END = bf.HISTORY_START, bf.DATA_AVAILABLE_THROUGH   # the whole history; change to a shorter range to test
print(f"days {START} .. {END}: {(END - START).days + 1} landing days x {len(bf.OPERATORS)} operators x 4 tables")

/home/javier/whizdom-churn/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


days 2026-06-18 .. 2026-10-04: 109 landing days x 2 operators x 4 tables


## 1. Volume in S3

One day, every table and operator: how many files and how many bytes. Only a listing, nothing is read.

In [2]:
with timer("1. list one day in S3"):
    s3, _ = daily_tables._clients()
    day = END - dt.timedelta(days=1)
    rows = []
    for op in bf.OPERATORS:
        for table, folder in bf.LANDING_TABLES.items():
            prefix = f"{bf.LANDING_LEVEL}/{op}/{folder}/{day.year}/{day.month:02d}/{day.day:02d}/"
            objs = [o for p in s3.get_paginator("list_objects_v2").paginate(Bucket=bf.BUCKET, Prefix=prefix) for o in p.get("Contents", [])]
            rows.append({"operator": op, "table": table, "files": len(objs), "mb": sum(o["Size"] for o in objs) / 1e6})
volume = pd.DataFrame(rows)
print(f"{day}: {volume['files'].sum()} files, {volume['mb'].sum() / 1e3:.2f} GB in S3 (full files)")
volume.round(1)

[timer] 1. list one day in S3: 1.7 s (0.0 min)
2026-10-03: 527 files, 2.70 GB in S3 (full files)


,operator,table,files,mb
0,primus,player,40,3.0
1,primus,bet,181,1763.3
2,primus,transaction,40,2.4
3,primus,bonus,40,4.2
4,secundus,player,38,1.9
5,secundus,bet,108,916.4
6,secundus,transaction,40,1.5
7,secundus,bonus,40,3.5


## 2. Build the daily tables

Only the missing days are read. The first run reads the whole history (about 110 days); after that, each run reads only the new days. `workers` = parallel file reads, `unit_workers` = tables/days at once (each `bet` day holds about 1 GB in memory while it is aggregated).

In [3]:
with timer("2. build daily tables"):
    units = daily_tables.build(START, END, workers=64, unit_workers=8)

built = units[units["status"] == "built"]
summary = built.groupby("table").agg(units=("day", "size"), files=("files", "sum"), source_rows=("source_rows", "sum"),
                                     daily_rows=("daily_rows", "sum"), mb_on_disk=("kb_on_disk", lambda s: s.sum() / 1e3),
                                     read_s=("read_s", "sum"), aggregate_s=("aggregate_s", "sum"))
print(units["status"].value_counts().to_string())
summary.round(1)

[   8/872 units] elapsed   0.2 min
[  16/872 units] elapsed   0.6 min
[  24/872 units] elapsed   0.8 min
[  32/872 units] elapsed   1.1 min
[  40/872 units] elapsed   1.3 min
[  48/872 units] elapsed   1.6 min
[  56/872 units] elapsed   1.8 min
[  64/872 units] elapsed   2.1 min
[  72/872 units] elapsed   2.4 min
[  80/872 units] elapsed   2.7 min
[  88/872 units] elapsed   2.9 min
[  96/872 units] elapsed   3.2 min
[ 104/872 units] elapsed   3.5 min
[ 112/872 units] elapsed   3.8 min
[ 120/872 units] elapsed   4.0 min
[ 128/872 units] elapsed   4.4 min
[ 136/872 units] elapsed   4.7 min
[ 144/872 units] elapsed   4.7 min
[ 152/872 units] elapsed   4.7 min
[ 160/872 units] elapsed   4.7 min
[ 168/872 units] elapsed   4.7 min
[ 176/872 units] elapsed   4.7 min
[ 184/872 units] elapsed   4.7 min
[ 192/872 units] elapsed   4.7 min
[ 200/872 units] elapsed   4.7 min
[ 208/872 units] elapsed   4.7 min
[ 216/872 units] elapsed   4.7 min
[ 224/872 units] elapsed   4.7 min
[ 232/872 units] ela

,units,files,source_rows,daily_rows,mb_on_disk,read_s,aggregate_s
table,,,,,,,
bet,130,16754,1381002253,1947737,44.1,1570.4,1632.6
bonus,130,5163,6524829,1496327,16.9,1099.9,468.4
player,130,5071,1709837,682328,4.3,1394.8,648.6
transaction,130,5171,7792426,1138648,9.4,1437.3,583.0


## 3. Disk used

In [4]:
def folder_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6

import alerts
print(f"daily tables:   {folder_mb(bf.DAILY_DIR):8.1f} MB  ({bf.DAILY_DIR.relative_to(PROJECT_ROOT)})")
print(f"alert profiles: {folder_mb(alerts.PROFILE_DIR):8.1f} MB  ({alerts.PROFILE_DIR.relative_to(PROJECT_ROOT)})")
print(f"free disk:      {shutil.disk_usage(PROJECT_ROOT).free / 1e9:8.1f} GB (inside WSL; see the note at the end)")

daily tables:      123.1 MB  (data/02_intermediate/daily_tables)
alert profiles:      3.9 MB  (data/02_intermediate/daily_profiles)
free disk:         882.1 GB (inside WSL; see the note at the end)


## 4. Features from the daily tables

The brand 64 snapshot of one cutoff, built from the daily tables only (no S3). When this was validated against the old S3 reads, every feature was identical (largest difference 8e-14, floating-point sums in a different order).

In [ ]:
CUTOFF, BRAND = dt.date(2026, 8, 5), 64
with timer("4. features from daily tables (one brand, one cutoff)"):
    snapshot = bf.raw_features(CUTOFF, BRAND)
print(f"{len(snapshot):,} players, {snapshot.shape[1]} columns")

## 5. Time summary

In [6]:
pd.Series(TIMINGS, name="seconds").round(1).to_frame().assign(minutes=lambda t: (t["seconds"] / 60).round(2))

,seconds,minutes
1. list one day in S3,1.7,0.03
2. build daily tables,1107.9,18.46
"4. features from daily tables (one brand, one cutoff)",4.6,0.08


## Note: reclaiming disk space in WSL

WSL keeps Linux files inside a virtual disk (`ext4.vhdx`) that **grows but never shrinks by itself**: deleting files inside WSL does not give the space back to Windows. This notebook writes very little (about 1.3 MB per day), but after deleting large files (old caches, DuckDB `.tmp` spills) the space only comes back after compacting the disk, from Windows (PowerShell as administrator):

```
wsl --shutdown
diskpart
select vdisk file="C:\Users\jherr\AppData\Local\Packages\CanonicalGroupLimited.Ubuntu22.04LTS_79rhkp1fndgsc\LocalState\ext4.vhdx"
attach vdisk readonly
compact vdisk
detach vdisk
exit
```